#Imports

In [ ]:
!pip install rdkit-pypi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 54.5 MB/s eta 0:00:00


In [ ]:
!pip install torch torchvision torchaudio

In [ ]:
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.0.1+cpu.html

!pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.0/494.0 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 11.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.9/750.9 kB 16.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.1/208.1 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from torch_geometric.nn import GINConv, global_mean_pool
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader as GeoDataLoader

/usr/local/lib/python3.11/dist-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /usr/local/lib/python3.11/dist-packages/torch_scatter/_version_cpu.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/usr/local/lib/python3.11/dist-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: /usr/local/lib/python3.11/dist-packages/torch_cluster/_version_cpu.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-cluster'. "
/usr/local/lib/python3.11/dist-packages/torch_geometric/typing.py:113: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: /usr/local/lib/python3.11/dist-packages/torch_spline_conv/_version_cpu.so: undefined symbol: _ZN3c1017RegisterOp

#Code

In [ ]:
def preprocess_genomic_data(file_path, feature_list):
    """
    Preprocess genomic data by selecting specific features and normalizing them.

    Parameters:
    - file_path: Path to the genomic data CSV file.
    - feature_list: List of genomic feature column names to select.

    Returns:
    - Normalized genomic data as a NumPy array.
    - The scaler object for inverse transformations if needed.
    """
    genomic_data = pd.read_csv(file_path)

    # Select only the relevant features from the feature list
    genomic_data_filtered = genomic_data[feature_list]

    # Standardize the features
    scaler = StandardScaler()
    genomic_data_normalized = scaler.fit_transform(genomic_data_filtered)

    return genomic_data_normalized, scaler

In [ ]:
from sklearn.preprocessing import MinMaxScaler
def smiles_to_graph(smiles, pIC50):
    """
    Convert a SMILES string to a graph representation.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Atom features
    atom_features = []
    for atom in mol.GetAtoms():
        atom_features.append(atom.GetAtomicNum())

    # Edge indices
    edge_indices = []
    for bond in mol.GetBonds():
        edge_indices.append([bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()])
        edge_indices.append([bond.GetEndAtomIdx(), bond.GetBeginAtomIdx()])

    # Convert to PyTorch tensors
    atom_features = torch.tensor(atom_features, dtype=torch.float).view(-1, 1)
    edge_indices = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
    logP = torch.tensor([pIC50], dtype=torch.float)

    # Create PyTorch Geometric Data object
    data = Data(x=atom_features, edge_index=edge_indices, y=pIC50)
    return data

def preprocess_drug_data(file_path):
    """
    Preprocess drug data by converting SMILES to graph representation.
    """
    drug_data = pd.read_csv(file_path, sep=',')

    # Fill NaN values in pIC50
    drug_data['pIC50'].fillna(drug_data['pIC50'].mean(), inplace=True)

    scaler = MinMaxScaler()
    drug_data['pIC50'] = scaler.fit_transform(drug_data[['pIC50']])

    drug_graphs = []
    for idx, row in drug_data.iterrows():
        graph = smiles_to_graph(row['SMILES'], row['pIC50'])
        if graph is not None:
            drug_graphs.append(graph)

    return drug_graphs

In [ ]:
# Dataset for Genomic Data
class GenomicDataset(Dataset):
    def __init__(self, genomic_data):
        """
        Dataset for genomic data.
        """
        self.genomic_data = genomic_data

    def __len__(self):
        return len(self.genomic_data)

    def __getitem__(self, idx):
        return torch.tensor(self.genomic_data[idx], dtype=torch.float)


# Dataset for Drug Data
class DrugDataset(Dataset):
    def __init__(self, drug_graphs):
        """
        Dataset for drug data (e.g., molecular graphs).
        """
        self.drug_graphs = drug_graphs

    def __len__(self):
        return len(self.drug_graphs)

    def __getitem__(self, idx):
        return self.drug_graphs[idx]


# Collate function for Genomic Data
def genomic_collate_fn(batch):
    """
    Collation function for genomic data.
    """
    genomic_features = torch.stack(batch)
    return genomic_features


# Collate function for Drug Data
def drug_collate_fn(batch):
    """
    Collation function for drug data.
    """
    drug_graphs = GeoDataLoader(batch, batch_size=len(batch), shuffle=False)
    return next(iter(drug_graphs))


# DataLoader creation for Genomic Data
def create_genomic_dataloader(genomic_data, batch_size=32):
    """
    Create DataLoader for genomic data.
    """
    dataset = GenomicDataset(genomic_data)
    return torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=genomic_collate_fn)


# DataLoader creation for Drug Data
def create_drug_dataloader(drug_graphs, batch_size=32):
    """
    Create DataLoader for drug data.
    """
    dataset = DrugDataset(drug_graphs)
    return torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=drug_collate_fn)

In [ ]:
genomic_features = [
        'cn_CEACAM6', 'cn_CD79A', 'cn_ATP1A3', 'cn_KCNN4', 'cn_CBLC',
        'cn_FOSB', 'cn_FOXA3', 'cn_KCNC3', 'cn_SPIB',
        'cn_KLK6', 'cn_KLK7', 'cn_KLK8', 'cn_KLK10', 'cn_KLK11', 'cn_KLK12', 'cn_KLK13', 'cn_KLK14',
        'cn_HAS1', 'cn_NLRP7', 'cn_TNNT1', 'cn_TNNI3', 'cn_SYT5', 'cn_COX6B2',
        'cn_NLRP5', 'cn_PEG3', 'cn_ZSCAN1', 'cn_DEFB132', 'cn_RSPO4',
        'cn_SIRPG', 'cn_TGM3', 'cn_ADAM33', 'cn_SPEF1', 'cn_PRND',
        'cn_CHGB', 'cn_FERMT1', 'cn_PAK7', 'cn_SNAP25', 'cn_FLRT3',
        'cn_PCSK2', 'cn_PTPRT', 'cn_BCAS1', 'cn_CYP24A1', 'cn_BMP7',
        'cn_PCK1', 'cn_ZBP1', 'cn_ZNF831', 'cn_EDN3',
        'mu_ATP10B', 'mu_FAT1', 'mu_FBN3', 'mu_FAT2', 'mu_MTOR',
        'mu_MDN1', 'mu_MXRA5', 'mu_MAP3K1',
        'mu_PTEN', 'mu_PIK3CA', 'mu_PCDH19',
        'mu_PLXNA4', 'mu_PTPRD', 'mu_PLCE1', 'mu_PCNT', 'mu_PCLO',
        'mu_PDE4DIP', 'mu_PKD1L1', 'mu_PCNXL2', 'mu_PRKDC', 'mu_PREX2'
]

# File paths
genomic_file = "/content/drive/MyDrive/genomic_data.csv"
drug_file = "/content/drive/MyDrive/drug_data.csv"

# Preprocess data
genomic_data, genomic_scaler = preprocess_genomic_data(genomic_file, genomic_features)
drug_graphs = preprocess_drug_data(drug_file)
print("Genomic Data Shape:", genomic_data.shape)

# Create DataLoaders
batch_size = 32
genomic_dataloader = create_genomic_dataloader(genomic_data, batch_size=batch_size)
drug_dataloader = create_drug_dataloader(drug_graphs, batch_size=batch_size)

for genomic_batch in genomic_dataloader:
    print("Genomic Batch Shape:", genomic_batch.shape)
    break

for drug_batch in drug_dataloader:
    print("Drug Batch:", drug_batch)
    break

<ipython-input-53-ee7a82d933e3>:37: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  drug_data['pIC50'].fillna(drug_data['pIC50'].mean(), inplace=True)


Genomic Data Shape: (705, 68)
Genomic Batch Shape: torch.Size([32, 68])
Drug Batch: DataBatch(x=[528, 1], edge_index=[2, 1094], y=[32], batch=[528], ptr=[33])


/usr/local/lib/python3.11/dist-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [ ]:
class DrugGNN(nn.Module):
    def __init__(self, node_features, hidden_dim):
        super(DrugGNN, self).__init__()
        # GIN Layers
        self.conv1 = GINConv(
            nn.Sequential(
                nn.Linear(node_features, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
        )
        self.conv2 = GINConv(
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
        )
        self.conv3 = GINConv(
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
        )

        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)

        self.dropout = nn.Dropout(0.2)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)

        # Global pooling
        x = global_mean_pool(x, batch)
        return x

In [ ]:
class GenomicNet(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(GenomicNet, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
class MultiTaskModel(nn.Module):
    def __init__(self, node_features, genomic_dim, hidden_dim=128):
        super(MultiTaskModel, self).__init__()

        self.drug_gnn = DrugGNN(node_features, hidden_dim)
        self.genomic_net = GenomicNet(genomic_dim, hidden_dim)

        # Shared layers for multi-task learning
        self.shared_layers = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Task-specific heads
        self.drug_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 1)
        )

        self.genomic_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, drug_data, genomic_data):
        # Drug features from GNN
        drug_features = self.drug_gnn(drug_data)

        # Genomic features from GenomicNet
        genomic_features = self.genomic_net(genomic_data)

        assert drug_features.shape[0] == genomic_features.shape[0], "Batch sizes do not match!"

        if drug_features.shape[1] != genomic_features.shape[1]:
            padding = torch.zeros(drug_features.shape[0], abs(drug_features.shape[1] - genomic_features.shape[1])).to(device)
            if drug_features.shape[1] > genomic_features.shape[1]:
                genomic_features = torch.cat((genomic_features, padding), dim=1)
            elif genomic_features.shape[1] > drug_features.shape[1]:
                drug_features = torch.cat((drug_features, padding), dim=1)

        # Combine features
        combined = torch.cat((drug_features, genomic_features), dim=1)

        # Shared feature representation
        shared_features = self.shared_layers(combined)

        # Task-specific predictions
        drug_pred = self.drug_head(shared_features)
        genomic_pred = self.genomic_head(shared_features)

        return drug_pred, genomic_pred

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import DataLoader as GeoDataLoader
from torch.utils.data import random_split
from torch_geometric.data import Batch

# Initialize the MultiTaskModel
node_features = drug_graphs[0].x.size(1)  # Assuming node features from drug data
genomic_dim = genomic_data.shape[1]  # Number of genomic features
hidden_dim = 128

model = MultiTaskModel(node_features=node_features, genomic_dim=genomic_dim, hidden_dim=hidden_dim)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Loss and optimizer
drug_criterion = nn.MSELoss()  # Regression task for drug labels
genomic_criterion = nn.MSELoss()  # For autoencoder-like task with genomic data
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Define the ReduceLROnPlateau scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5, verbose=True)

# Split DataLoader function
def split_dataloader(dataloader, split_ratios):
    dataset = dataloader.dataset
    train_len = int(split_ratios[0] * len(dataset))
    val_len = int(split_ratios[1] * len(dataset))
    test_len = len(dataset) - train_len - val_len
    train_data, val_data, test_data = random_split(dataset, [train_len, val_len, test_len])

    # Use GeoDataLoader for graph data
    train_loader = GeoDataLoader(train_data, batch_size=dataloader.batch_size, shuffle=True)
    val_loader = GeoDataLoader(val_data, batch_size=dataloader.batch_size, shuffle=False)
    test_loader = GeoDataLoader(test_data, batch_size=dataloader.batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

# Split genomic and drug DataLoaders (use GeoDataLoader for drug data)
genomic_train_loader, genomic_val_loader, genomic_test_loader = split_dataloader(genomic_dataloader, [0.7, 0.15, 0.15])
drug_train_loader, drug_val_loader, drug_test_loader = split_dataloader(drug_dataloader, [0.7, 0.15, 0.15])

# Combined training and validation loop with ReduceLROnPlateau
def train_and_validate(model, genomic_train_loader, drug_train_loader, genomic_val_loader, drug_val_loader, optimizer, drug_criterion, genomic_criterion, scheduler, device, num_epochs):
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        # Training Phase
        for genomic_batch, drug_batch in zip(genomic_train_loader, drug_train_loader):
            batch_size = min(genomic_batch.size(0), drug_batch.size(0))  # Use the smaller batch size
            genomic_batch = genomic_batch[:batch_size]
            drug_batch = drug_batch[:batch_size]

            genomic_batch = genomic_batch.to(device)
            drug_batch = Batch.from_data_list(drug_batch).to(device)

            optimizer.zero_grad()
            drug_pred, genomic_pred = model(drug_batch, genomic_batch)

            # Drug labels are in the 'pIC50' column
            drug_labels = drug_batch.y
            genomic_labels = genomic_batch

            # Calculate loss
            drug_loss = drug_criterion(drug_pred, drug_labels)
            genomic_loss = genomic_criterion(genomic_pred, genomic_labels)  # Autoencoder-style loss

            loss = drug_loss + genomic_loss
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        avg_train_loss = train_loss / len(genomic_train_loader)

        # Validation Phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for genomic_batch, drug_batch in zip(genomic_val_loader, drug_val_loader):
                batch_size = min(genomic_batch.size(0), drug_batch.size(0))  # Use the smaller batch size
                genomic_batch = genomic_batch[:batch_size]
                drug_batch = drug_batch[:batch_size]

                genomic_batch = genomic_batch.to(device)
                drug_batch = Batch.from_data_list(drug_batch).to(device)

                drug_pred, genomic_pred = model(drug_batch, genomic_batch)

                # Drug labels are in the 'pIC50' column
                drug_labels = drug_batch.y
                genomic_labels = genomic_batch

                # Calculate loss
                drug_loss = drug_criterion(drug_pred, drug_labels)
                genomic_loss = genomic_criterion(genomic_pred, genomic_labels)   # Autoencoder-style loss

                loss = drug_loss + genomic_loss
                val_loss += loss.item()

        avg_val_loss = val_loss / len(genomic_val_loader)
        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}")

        # Step the scheduler based on validation loss
        scheduler.step(avg_val_loss)

# Call the train and validate function
train_and_validate(model, genomic_train_loader, drug_train_loader, genomic_val_loader, drug_val_loader, optimizer, drug_criterion, genomic_criterion, scheduler, device, num_epochs=10)

Epoch 1/10, Train Loss: 0.9802, Validation Loss: 0.9242
Epoch 2/10, Train Loss: 0.8840, Validation Loss: 0.8534
Epoch 3/10, Train Loss: 0.8950, Validation Loss: 0.8504
Epoch 4/10, Train Loss: 0.8893, Validation Loss: 0.8389
Epoch 5/10, Train Loss: 0.8860, Validation Loss: 0.8484
Epoch 6/10, Train Loss: 0.9005, Validation Loss: 0.8530
Epoch 7/10, Train Loss: 0.8986, Validation Loss: 0.8476
Epoch 8/10, Train Loss: 0.8770, Validation Loss: 0.8349
Epoch 9/10, Train Loss: 0.8818, Validation Loss: 0.8650
Epoch 10/10, Train Loss: 0.8498, Validation Loss: 0.8461


In [ ]:
# Test the model
def test(model, genomic_loader, drug_loader, device):
    total_drug_loss = 0
    total_genomic_loss = 0
    total_batches = 0
    model.eval()
    with torch.no_grad():
        for genomic_batch, drug_batch in zip(genomic_loader, drug_loader):
            batch_size = min(genomic_batch.size(0), drug_batch.size(0))  # Use the smaller batch size
            genomic_batch = genomic_batch[:batch_size]
            drug_batch = drug_batch[:batch_size]
            genomic_batch = genomic_batch.to(device)

            # Ensure `drug_batch` is correctly formatted
            if isinstance(drug_batch, Batch):
                drug_batch = drug_batch.to(device)
            else:
                drug_batch = Batch.from_data_list([data for data in drug_batch]).to(device)

            # Make predictions
            drug_pred, genomic_pred = model(drug_batch, genomic_batch)
            drug_labels = drug_batch.y
            genomic_labels = genomic_batch

            drug_loss = drug_criterion(drug_pred, drug_labels)
            genomic_loss = genomic_criterion(genomic_pred, genomic_labels)

            total_drug_loss += drug_loss.item()
            total_genomic_loss += genomic_loss.item()
            total_batches += 1

            # Print batch predictions and losses (optional)
            print(f"Batch {total_batches}:")
            print("Drug Loss:", drug_loss.item())
            print("Genomic Loss:", genomic_loss.item())
    avg_drug_loss = total_drug_loss / total_batches
    avg_genomic_loss = total_genomic_loss / total_batches
    print("\nTest Results:")
    print(f"Average Drug Loss: {avg_drug_loss:.4f}")
    print(f"Average Genomic Loss: {avg_genomic_loss:.4f}")

test(model, genomic_test_loader, drug_test_loader, device)

Batch 1:
Drug Loss: 0.037776604294776917
Genomic Loss: 0.9033658504486084
Batch 2:
Drug Loss: 0.015069923363626003
Genomic Loss: 0.8223863244056702
Batch 3:
Drug Loss: 0.0753210186958313
Genomic Loss: 0.6833887696266174
Batch 4:
Drug Loss: 0.07911588251590729
Genomic Loss: 0.8242080807685852

Test Results:
Average Drug Loss: 0.0518
Average Genomic Loss: 0.8083


#Old

In [ ]:
def display_results(predictions):
    print("\nResults:")
    print(f"{'Drug':<10}{'Compatibility':<15}{'Toxicity':<10}{'Confidence (%)':<15}{'Explanation':<30}{'Actions':<20}")
    print("-" * 100)

    for result in predictions:
        drug = result['drug']
        compatibility = "Compatible" if result['compatibility'] else "Not compatible"
        toxicity = "Non-toxic" if result['toxicity'] else "Toxic"
        confidence = result['confidence']
        explanation = result['explanation']
        actions = result['actions']

        print(f"{drug:<10}{compatibility:<15}{toxicity:<10}{confidence:<15}{explanation:<30}{actions:<20}")

In [ ]:
def evaluate_model(model_path, test_loader, device):
    # Load the saved model
    model = MultiTaskModel(
        node_features=6,  # Number of atom features
        edge_features=3,  # Number of bond features
        genomic_dim=len(test_loader.dataset.genomic_feature_names),
        hidden_dim=128  # Hidden dimension for the networks
    )
    model.load_state_dict(torch.load(model_path))
    model.to(device)
    model.eval()

    predictions = []

    with torch.no_grad():
        for batch in test_loader:
            drug_data = batch['drug'].to(device)
            genomic_data = batch['genomic'].to(device)

            drug_pred, genomic_pred = model(drug_data, genomic_data)

            for i in range(len(drug_pred)):
                predictions.append({
                    'drug': batch['drug_name'][i],
                    'compatibility': bool(drug_pred[i] > 0.5),
                    'toxicity': bool(genomic_pred[i] < 0.3),
                    'confidence': int(torch.sigmoid(drug_pred[i]).item() * 100),
                    'explanation': "Prediction based on gene expression",
                    'actions': "Recommended" if drug_pred[i] > 0.5 else "Avoid use"
                })

    display_results(predictions)

In [ ]:
def process_test_data(drug_data_path, genomic_data_path, model_path, device):
    # Load drug data
    drug_data = pd.read_csv(drug_data_path)
    drug_data = drug_data[['SMILES', 'pIC50']]

    # Load genomic data
    genomic_data = pd.read_csv(genomic_data_path)
    genomic_features = [
        'cn_CEACAM6', 'cn_CD79A', 'cn_ATP1A3', 'cn_KCNN4', 'cn_CBLC',
        'cn_FOSB', 'cn_FOXA3', 'cn_KCNC3', 'cn_SPIB',
        'cn_KLK6', 'cn_KLK7', 'cn_KLK8', 'cn_KLK10', 'cn_KLK11', 'cn_KLK12', 'cn_KLK13', 'cn_KLK14',
        'cn_HAS1', 'cn_NLRP7', 'cn_TNNT1', 'cn_TNNI3', 'cn_SYT5', 'cn_COX6B2',
        'cn_NLRP5', 'cn_PEG3', 'cn_ZSCAN1', 'cn_DEFB132', 'cn_RSPO4',
        'cn_SIRPG', 'cn_TGM3', 'cn_ADAM33', 'cn_SPEF1', 'cn_PRND',
        'cn_CHGB', 'cn_FERMT1', 'cn_PAK7', 'cn_SNAP25', 'cn_FLRT3',
        'cn_PCSK2', 'cn_PTPRT', 'cn_BCAS1', 'cn_CYP24A1', 'cn_BMP7',
        'cn_PCK1', 'cn_ZBP1', 'cn_ZNF831', 'cn_EDN3',
        'mu_ATP10B', 'mu_FAT1', 'mu_FBN3', 'mu_FAT2', 'mu_MTOR',
        'mu_MDN1', 'mu_MXRA5', 'mu_MAP3K1',
        'mu_PTEN', 'mu_PIK3CA', 'mu_PCDH19',
        'mu_PLXNA4', 'mu_PTPRD', 'mu_PLCE1', 'mu_PCNT', 'mu_PCLO',
        'mu_PDE4DIP', 'mu_PKD1L1', 'mu_PCNXL2', 'mu_PRKDC', 'mu_PREX2'
    ]

    # Check for missing features
    missing_features = [col for col in genomic_features if col not in genomic_data.columns]
    if missing_features:
        raise ValueError(f"Missing genomic features: {missing_features}")

    # Create molecular dataset
    molecular_dataset = MolecularDataset(
        smiles_data=drug_data['SMILES'].values,
        labels=drug_data['pIC50'].values
    )

    # Standardize genomic features
    scaler = StandardScaler()
    genomic_features_scaled = scaler.fit_transform(genomic_data[genomic_features])

    # Create test dataset and loader
    test_dataset = CombinedDataset(
        drug_data=[molecular_dataset[i] for i in range(len(molecular_dataset))],
        genomic_data=torch.FloatTensor(genomic_features_scaled),
        drug_labels=torch.FloatTensor(molecular_dataset.labels),
        genomic_labels=torch.FloatTensor(genomic_features_scaled)
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False,
        collate_fn=collate_batch
    )

    # Evaluate the model
    evaluate_model(model_path, test_loader, device)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    # Paths to data
    drug_data_path = "drug_data_pred.csv"
    genomic_data_path = "genomic_data_pred.csv"
    model_path = "best_model.pth"

    # Process test data and evaluate
    process_test_data(drug_data_path, genomic_data_path, model_path, device)
except Exception as e:
    print(f"Error during execution: {e}")
    raise